In [ ]:
#YEARLY COGNITIVON DDIVERGENCE ANALYSIS - COMPOSITE COGNITION

"""
Sets up the YEARLY DIVERGENCE analysis (pooled composite cognition).

This design replicates the design by, You et al (2024)s. 

DESIGN (nested case-control, per time window):
- Cases: participants with incident dementia. A case's test contributes to the
  window matching its own lead time (years from test to diagnosis).
- Controls: participants who never develop dementia. Controls have no diagnosis
  date, so each control is assigned a comparison time. We mirror cases by
  matching: for each case-window we draw controls and give them the same
  'years before index' as the case (an arbitrary index date on their own
  follow-up). This solves the 'controls have no time-to-diagnosis' problem the
  same way You et al did.
- In each window we fit: dementia_case ~ cognition + age, giving an OR per
  1 SD worse cognition for tests taken that many years before diagnosis.

The earliest window whose OR significantly differs from 1 is the divergence
horizon: the point at which future cases are already measurably impaired.

POOLED: one composite score per visit (mean of direction-adjusted z across the
tests done at that visit, higher = worse).

MATCHING: each case-window observation is matched to N_CONTROLS controls on
age (+/- AGE_CALIPER) and sex, following You et al's matched design. Set
N_CONTROLS and AGE_CALIPER below.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns

#minimal plotting size
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.linewidth'] = 0.8

output_dir = 'divergence_yearly'
os.makedirs(output_dir, exist_ok=True)

N_CONTROLS  = 10     # controls per case-observation (You used 10)
AGE_CALIPER = 2.0    # years, for age matching
RNG_SEED    = 42

#load data
data = pd.read_csv('cv_results_all_tests_merged_with_groups.csv')
data = data.rename(columns={'Sex': 'sex', 'Edu': 'edu'})
data = data[data['test_name'] != 'PMem'].copy()

n_prev = (data['prevalent'] == 1).sum()
data = data[data['prevalent'] == 0].copy()
print(f"Dropped {n_prev:,} prevalent observations")

test_direction = {
    'PMat': -1, 'RT': -1, 'FI': 1, 'NM': 1, 'SD': 1, 'TMA': -1,
    'TMB': -1, 'MP': 1, 'PAL': 1, 'PV': 1, 'TR': 1, 'BL': 1
}

# higher = worse performance
data['z_score_adj'] = -(data['z_score'] * data['test_name'].map(test_direction))

# Composite per visit, keyed on instance (not rounded age)
visit_df = (data.groupby(['eid', 'instance']).agg(
    z_score       = ('z_score_adj', 'mean'),
    n_tests       = ('z_score_adj', 'size'),
    age_at_test   = ('age_at_test', 'mean'),
    time_to_event = ('time_to_event', 'first'),
    event         = ('event', 'first'),
    sex           = ('sex', 'first'),
    edu           = ('edu', 'first'),
).reset_index())

print("\nTests contributing per visit:")
print(visit_df['n_tests'].value_counts().sort_index().to_string())

visit_df = visit_df[visit_df['time_to_event'] > 0].dropna(
    subset=['z_score', 'age_at_test', 'time_to_event', 'event', 'sex']).copy()

COMP_SD = visit_df['z_score'].std(ddof=1)
visit_df['z_score'] = visit_df['z_score'] / COMP_SD
print(f"\nComposite raw SD = {COMP_SD:.3f} (divided out; composite now SD 1)")

windows = list(range(1, 26))
print(f"\nVisits = {len(visit_df):,}  |  participants = {visit_df['eid'].nunique():,}")
print(f"Case visits = {int(visit_df['event'].sum()):,}  |  "
      f"control visits = {int((visit_df['event']==0).sum()):,}")

visit_df = visit_df[visit_df['time_to_event'] > 0].dropna(
    subset=['z_score', 'age_at_test', 'time_to_event', 'event']).copy()

if 'sex' not in visit_df.columns:
    print("WARNING: no 'sex' column found - matching on age only. "
          "Add sex to the source file to match You et al exactly.")
    visit_df['sex'] = 0

#Time windows before diagnosis (years). 0-1, 1-2, ... 14-15
windows = list(range(1, 26))

print(f"\nVisits = {len(visit_df):,}  |  participants = {visit_df['eid'].nunique():,}")
print(f"Case visits = {int(visit_df['event'].sum()):,}  |  "
      f"control visits = {int((visit_df['event']==0).sum()):,}")

In [ ]:
#Build the matched per-window dataset (Python)
"""
For each case VISIT, its window = floor(time_to_event) + 1 (i.e. a visit 3.4
years before diagnosis falls in window 4, meaning '3-4 years before').
Each case visit is matched to N_CONTROLS control visits drawn from controls
whose own visit shares the same window position on their follow-up, within the
age caliper and same sex. Controls are assigned the case's window as their
'years before index'.

The result is a long table: one row per (case or matched control) x window,
with a binary 'is_case' outcome, ready for per-window logistic regression.
"""
rng = np.random.default_rng(RNG_SEED)

visit_df = visit_df.copy()
visit_df['window'] = np.floor(visit_df['time_to_event']).astype(int) + 1
visit_df = visit_df[visit_df['window'].between(1, max(windows))]

cases    = visit_df[visit_df['event'] == 1].copy()
controls = visit_df[visit_df['event'] == 0].copy()

matched_rows = []
mid = 0
for w in windows:
    cw = cases[cases['window'] == w]
    # one visit per control person, still at risk when the case was diagnosed
    pool = (controls[controls['time_to_event'] >= (w - 1)]
            .sample(frac=1, random_state=RNG_SEED)
            .drop_duplicates('eid'))
    if len(cw) == 0 or len(pool) == 0:
        continue
    pool_age, pool_sex = pool['age_at_test'].values, pool['sex'].values
    pool_idx = pool.index.values

    for _, case in cw.iterrows():
        elig = pool_idx[(np.abs(pool_age - case['age_at_test']) <= AGE_CALIPER)
                        & (pool_sex == case['sex'])]
        if len(elig) == 0:
            continue
        take = rng.choice(elig, size=min(N_CONTROLS, len(elig)), replace=False)
        mid += 1
        matched_rows.append({'match_id': mid, 'window': w, 'is_case': 1,
                             'z_score': case['z_score'],
                             'age_at_test': case['age_at_test'],
                             'edu': case['edu'], 'eid': case['eid']})
        for ci in take:
            cr = controls.loc[ci]
            matched_rows.append({'match_id': mid, 'window': w, 'is_case': 0,
                                 'z_score': cr['z_score'],
                                 'age_at_test': cr['age_at_test'],
                                 'edu': cr['edu'], 'eid': cr['eid']})

matched = pd.DataFrame(matched_rows)
print(f"Matched dataset: {len(matched):,} rows across {matched['window'].nunique()} windows")
chk = matched.groupby('window').agg(cases=('is_case','sum'), total=('is_case','count'),
                                    people=('eid','nunique'))
chk['rows_per_person'] = chk['total'] / chk['people']
print(chk)

In [ ]:
#Per-window logistic regression in R
"""
-i matched -i windows: imports to R
-o all_results: per-window OR back to Python

For each window w:
  Fit: is_case ~ z_score + age_at_test  (logistic)
  z_score is direction-adjusted (higher = worse), so OR > 1 means worse
  cognition at that lead time is associated with being a future case.
"""
%load_ext rpy2.ipython

In [ ]:
%%R -i matched -i windows -o all_results

library(broom)
library(dplyr)
library(survival)

all_results <- list()
row_idx <- 1

for (w in windows) {
  wd <- matched %>% filter(window == w)
  n_cases <- sum(wd$is_case)
  if (n_cases < 5 || nrow(wd) < 2) {
      cat( " window", w, "SKIPPED: (",n_cases, "cases in window)\n")
      next

  }
      
  fit <- tryCatch(
    clogit(is_case ~ z_score + age_at_test + strata(match_id), data = wd),
    error = function(e) { cat("  window", w, "ERROR\n"); return(NULL) })
  if (is.null(fit)) next

  tidy_fit <- tidy(fit, conf.int = TRUE, exponentiate = TRUE)
  z_row <- tidy_fit[tidy_fit$term == "z_score", ]

  cat("  window", w, " OR=", round(z_row$estimate, 3),
      " cases=", n_cases, "\n")

  all_results[[row_idx]] <- data.frame(
    window   = w,
    n_total  = nrow(wd),
    n_cases  = n_cases,
    z_or     = z_row$estimate,
    z_ci_low = z_row$conf.low,
    z_ci_high= z_row$conf.high,
    z_pval   = z_row$p.value
  )
  row_idx <- row_idx + 1
}

all_results <- do.call(rbind, all_results)
cat("\nTotal windows fit:", nrow(all_results), "\n")

In [ ]:
#Divergence horizon and figure
"""
Divergence horizon: the earliest (largest-w) window whose OR lower CI still
excludes 1. The OR trajectory across windows is the divergence curve - the
mirror image of a cognitive trajectory plot, in odds-ratio space (as in You
et al Fig 3).
"""
all_results = all_results.sort_values('window').reset_index(drop=True)

sig = all_results[all_results['z_ci_low'] > 1]
horizon = int(sig['window'].max()) if len(sig) else 0
print(f"Divergence horizon: {horizon} years before diagnosis")
print(all_results[['window', 'n_cases', 'z_or', 'z_ci_low', 'z_ci_high']].round(3).to_string(index=False))

all_results.to_csv(f'{output_dir}/divergence_results_pooled.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 6))
ax.fill_between(all_results['window'], all_results['z_ci_low'],
                all_results['z_ci_high'], alpha=0.25, color='black')
ax.plot(all_results['window'], all_results['z_or'],
        marker='o', markersize=5, color='black', linewidth=2)
ax.axhline(y=1, color='darkgrey', linestyle='--', linewidth=1)
if horizon > 0:
    ax.axvline(x=horizon, color='red', linestyle=':', linewidth=1.5)
ax.invert_xaxis()  # diagnosis on the right, far past on the left
ax.set_xlabel('Years before diagnosis', fontsize=11)
ax.set_ylabel('OR per 1 SD worse cognition (case vs matched control)', fontsize=11)
ax.set_title('Yearly divergence (composite cognition): when do future cases diverge?',
             fontsize=12)
ax.set_xticks(all_results['window'])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{output_dir}/divergence_pooled.png', dpi=800, bbox_inches='tight')
plt.show()